In [11]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.models import load_model

In [12]:
model = load_model("model.keras")
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
df = pd.read_csv("Dataset/household_power_consumption.txt", sep=";", na_values="?", low_memory=False)
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df["Global_active_power"] = pd.to_numeric(df["Global_active_power"], errors="coerce")
df = df[["Datetime", "Global_active_power"]]
df = df.set_index("Datetime")
df = df.sort_index()
df["Global_active_power"] = df["Global_active_power"].interpolate(method="time")
df = df.dropna()
df = df.resample("30min").mean()
df = df.dropna()
df.head()

,Global_active_power
Datetime,
2006-12-16 17:00:00,4.587333
2006-12-16 17:30:00,4.150000
2006-12-16 18:00:00,3.944800
2006-12-16 18:30:00,3.319600
2006-12-16 19:00:00,3.464400


In [14]:
SEQUENCE_LENGTH = 60
latest_data = df["Global_active_power"].values[-SEQUENCE_LENGTH:]
print("Number of observations:", len(latest_data))

Number of observations: 60


In [15]:
latest_scaled = scaler.transform(latest_data.reshape(-1, 1))

In [16]:
X_input = latest_scaled.reshape(1, SEQUENCE_LENGTH, 1)
print("Input shape:", X_input.shape)

Input shape: (1, 60, 1)


In [17]:
prediction_scaled = model.predict(X_input)
prediction = scaler.inverse_transform(prediction_scaled)
predicted_power = prediction[0][0]
print(f"Predicted next 30-minute power consumption: " f"{predicted_power:.4f} kW")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step
Predicted next 30-minute power consumption: 1.0833 kW


In [18]:
print("Latest 10 observations:")
print(df.tail(10))
print(f"\nPredicted Next Value: " f"{predicted_power:.4f} kW")

Latest 10 observations:
                     Global_active_power
Datetime                                
2010-11-26 16:30:00             1.284200
2010-11-26 17:00:00             1.619533
2010-11-26 17:30:00             1.832267
2010-11-26 18:00:00             1.409000
2010-11-26 18:30:00             1.737933
2010-11-26 19:00:00             1.591400
2010-11-26 19:30:00             1.727267
2010-11-26 20:00:00             1.356733
2010-11-26 20:30:00             0.970667
2010-11-26 21:00:00             0.934667

Predicted Next Value: 1.0833 kW
